# Broadcast CV pipeline run

Runs roboflow/sports soccer modes on a 20-second broadcast clip, times each mode, and saves outputs for review.

Kernel: Colab with **GPU (T4)**.

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# 1. Clone roboflow/sports and this repo
REPO_URL = "https://github.com/varunrout/broadcast-cv-pipeline.git"  # change if needed

%cd /content
!git clone -q https://github.com/roboflow/sports.git
!git clone -q {REPO_URL} pipeline

/content


In [3]:
# 2. Install
%cd /content/sports
!pip install -q -e .
%cd /content/sports/examples/soccer
!pip install -q -r requirements.txt
!pip install -q -r /content/pipeline/requirements.txt

/content/sports
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sports (pyproject.toml) ... done
/content/sports/examples/soccer


In [ ]:
# 2b. Make sure torch can see the GPU
# The soccer requirements can replace Colab's CUDA torch with a CPU-only build.
import subprocess
check = 'import torch; print(torch.__version__, torch.cuda.is_available())'
print(subprocess.run(['python', '-c', check], capture_output=True, text=True).stdout)

# If the line above ends in False, reinstall the CUDA build:
!pip uninstall -q -y torch torchvision
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu126
print(subprocess.run(['python', '-c', check], capture_output=True, text=True).stdout)

In [4]:
# 3. Download models and sample videos (~400 MB)
%cd /content/sports/examples/soccer
!bash setup.sh
!ls -lh data

/content/sports/examples/soccer
Downloading...
From (original): https://drive.google.com/uc?id=1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V
From (redirected): https://drive.google.com/uc?id=1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V&confirm=t&uuid=cc2e0f0a-f3be-4c3e-92f2-7a677b80c04d
To: /content/sports/examples/soccer/data/football-ball-detection.pt
100% 137M/137M [00:02<00:00, 66.6MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q
From (redirected): https://drive.google.com/uc?id=17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q&confirm=t&uuid=17d3ade6-a35f-41a3-b5d2-ea1d890fd7bf
To: /content/sports/examples/soccer/data/football-player-detection.pt
100% 137M/137M [00:00<00:00, 230MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1Ma5Kt86tgpdjCTKfum79YMgNnSjcoOyf
From (redirected): https://drive.google.com/uc?id=1Ma5Kt86tgpdjCTKfum79YMgNnSjcoOyf&confirm=t&uuid=094caf8d-d9d2-4332-a05c-b54c1c66d8ea
To: /content/sports/examples/soccer/data/football-pi

In [5]:
# 4. Trim to a 20-second clip
!ffmpeg -y -loglevel error -i /content/sports/examples/soccer/data/2e57b9_0.mp4 -t 20 -c copy /content/sports/examples/soccer/data/clip.mp4

In [6]:
# 5. Run all modes and log runtimes
%cd /content/pipeline
!python src/run_modes.py --soccer-dir /content/sports/examples/soccer --clip data/clip.mp4 --out-dir outputs --device cuda

/content/pipeline
[PITCH_DETECTION] FAILED
^^^^^^^
  File "/content/sports/examples/soccer/main.py", line 172, in run_pitch_detection
    pitch_detection_model = YOLO(PITCH_DETECTION_MODEL_PATH).to(device=device)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1388, in to
    return self._apply(convert)
           ~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/ultralytics/engine/model.py", line 998, in _apply
    super()._apply(apply)
    ~~~~~~~~~~~~~~^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 933, in _apply
    module._apply(fn)
    ~~~~~~~~~~~~~^^^^
  File "/usr/local/lib/python3.13/dist-packages/ultralytics/nn/tasks.py", line 314, in _apply
    super()._apply(fn)
    ~~~~~~~~~~~~~~^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 933, in _apply
    module._apply(fn)
    ~~~~~~~~~~~~~^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/

In [7]:
# 6. fps and real-time ratio per mode
!python src/benchmark.py --clip /content/sports/examples/soccer/data/clip.mp4

Clip: 502 frames at 25.0 fps (20.1s)

PITCH_DETECTION          0.00 fps  (0.00x real time)
PLAYER_TRACKING          0.00 fps  (0.00x real time)
TEAM_CLASSIFICATION      0.00 fps  (0.00x real time)
RADAR                    0.00 fps  (0.00x real time)

Benchmark written to results/benchmark.csv


In [8]:
# 7. Re-encode outputs so they play inline
import glob, os
for src in sorted(glob.glob('/content/pipeline/outputs/out_*.mp4')):
    dst = src.replace('out_', 'view_')
    os.system(f'ffmpeg -y -loglevel error -i {src} -vcodec libx264 -preset veryfast -crf 28 {dst}')
!ls -lh /content/pipeline/outputs

total 16K
-rw-r--r-- 1 root root 258 Nov 14 15:18 out_pitch_detection.mp4
-rw-r--r-- 1 root root 258 Nov 14 15:18 out_player_tracking.mp4
-rw-r--r-- 1 root root 258 Nov 14 15:19 out_radar.mp4
-rw-r--r-- 1 root root 258 Nov 14 15:19 out_team_classification.mp4


In [9]:
# 8. Watch one mode at a time
from IPython.display import Video
MODE_TO_VIEW = 'player_tracking'  # pitch_detection | player_tracking | team_classification | radar
Video(f'/content/pipeline/outputs/view_{MODE_TO_VIEW}.mp4', embed=True, width=800)

Fill in `results/failure_modes.md` while watching. Download `results/timings.csv` and `results/benchmark.csv` back into the repo.